In [1]:
"""
Layer 2 GA only model Training

Author: Tan Yi Feng
Student ID: 23WMR14766
"""
import pandas as pd
import numpy as np
import time
import psutil
import os
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight

from sklearn_genetic import GAFeatureSelectionCV, GASearchCV
from sklearn_genetic.space import Integer, Categorical
from deap import creator

warnings.filterwarnings("ignore", category=RuntimeWarning)
from pathlib import Path
import joblib
import pickle
import pandas as pd


SCRIPT_DIR = Path.cwd()


DATA_PATH = SCRIPT_DIR / 'data' / 'FYP_Processed_Data'/'balanced_train_data.csv'

print("Loading data...")
df = pd.read_csv(DATA_PATH)


print("Filtering out normal traffic (Label 0) for Dual-Layer Architecture...")
df = df[df['label'] != 0].copy() 

print("Merging attack classes...")

label_merge_map = {
    1: 1,   # arp spoof
    2: 2, 3: 3, 4: 2, 5: 3, # MQTT
    6: 4,                   # Malformed
    7: 5, 8: 6, 9: 5, 10: 5, # Recon
    11: 7, 15: 7,           # ICMP Flood
    12: 8, 16: 8,           # SYN Flood
    13: 9, 17: 9,           # TCP Flood
    14: 10, 18: 10          # UDP Flood
}

df['label'] = df['label'].map(label_merge_map)

print("Attack-only class distribution:")
print(df['label'].value_counts().sort_index())


df.replace([np.inf, -np.inf], np.nan, inplace=True)
X = df.drop(columns=['label']).select_dtypes(include=['number'])
y = df['label']

X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns)
y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


classes = np.unique(y_train)
base_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, base_weights))


penalty_classes = [1, 2, 3, 4, 5, 6] 
for cls in penalty_classes:
    if cls in class_weight_dict:
        class_weight_dict[cls] *= 2.0 

print("Class weights:")
print(class_weight_dict)


for cls in ["FitnessMax", "Individual"]:
    if hasattr(creator, cls):
        delattr(creator, cls)


print("\n=== STAGE 1: GA FEATURE SELECTION ===")

fs_pipeline = RandomForestClassifier(
    n_estimators=150,
    max_depth=25,
    random_state=42,
    n_jobs=-1,
    class_weight=class_weight_dict
)

ga_fs = GAFeatureSelectionCV(
    estimator=fs_pipeline,
    scoring='f1_weighted',
    cv=3,
    population_size=30,
    generations=30,
    crossover_probability=0.8,
    mutation_probability=0.01,
    n_jobs=-1,
    verbose=True,
    keep_top_k=5
)

t0 = time.time()
ga_fs.fit(X_train, y_train)
fs_time = time.time() - t0

selected_features = X_train.columns[ga_fs.support_]
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

print(f"Selected {len(selected_features)} / {X_train.shape[1]} features")


print("\n=== STAGE 2: GA HYPERPARAMETER OPTIMIZATION ===")

rf_pipeline = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight=class_weight_dict
)

param_grid = {
    'n_estimators': Integer(100, 300),
    'max_depth': Integer(10, 40),
    'min_samples_split': Integer(2, 10),
    'min_samples_leaf': Integer(1, 5),
    'bootstrap': Categorical([True, False]),
    'criterion': Categorical(['gini', 'entropy'])
}

ga_hp = GASearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=3,
    population_size=30,
    generations=30,
    crossover_probability=0.8,
    mutation_probability=0.01,
    n_jobs=-1,
    verbose=True,
    keep_top_k=5
)

process = psutil.Process(os.getpid())
cpu_start = psutil.cpu_percent(interval=None)
mem_start = process.memory_info().rss / (1024 ** 2)

t1 = time.time()
ga_hp.fit(X_train_sel, y_train)
hp_time = time.time() - t1

cpu_end = psutil.cpu_percent(interval=None)
mem_end = process.memory_info().rss / (1024 ** 2)


best_model = ga_hp.best_estimator_

pred_start = time.time()
y_pred = best_model.predict(X_test_sel)
pred_time = time.time() - pred_start

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print("\n=== RESOURCE USAGE ===")
print(f"Feature Selection Time : {fs_time:.2f} s")
print(f"Hyperparameter Time   : {hp_time:.2f} s")
print(f"CPU Usage             : {cpu_start:.2f}% → {cpu_end:.2f}%")
print(f"Memory Used           : {mem_end - mem_start:.2f} MB")
print(f"Inference Time        : {pred_time:.4f} s")

print("\n=== FINAL PERFORMANCE ===")
print("Best Hyperparameters:")
print(ga_hp.best_params_)

print(f"Accuracy     : {acc:.4f}")
print(f"Macro F1     : {f1_macro:.4f}")
print(f"Weighted F1  : {f1_weighted:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Loading data...
Filtering out normal traffic (Label 0) for Dual-Layer Architecture...
Merging attack classes...
Attack-only class distribution:
label
1      800
2     1600
3     1600
4      800
5     2400
6      559
7     3600
8     2400
9     2400
10    3600
Name: count, dtype: int64
Class weights:
{np.int64(0): np.float64(2.46984375), np.int64(1): np.float64(2.46984375), np.int64(2): np.float64(2.46984375), np.int64(3): np.float64(4.9396875), np.int64(4): np.float64(1.6465625), np.int64(5): np.float64(7.07248322147651), np.int64(6): np.float64(1.0977083333333333), np.int64(7): np.float64(0.82328125), np.int64(8): np.float64(0.82328125), np.int64(9): np.float64(0.5488541666666666)}

=== STAGE 1: GA FEATURE SELECTION ===
gen	nevals	fitness 	fitness_std	fitness_max	fitness_min
0  	30    	0.907232	0.0744022  	0.969452   	0.678158   
1  	49    	0.958798	0.0224338  	0.969965   	0.895773   
2  	51    	0.969069	0.000765612	0.969965   	0.966735   
3  	46    	0.969687	0.00041926 	0.970547   	0

In [2]:
import os
from pathlib import Path
import joblib
import pickle
import pandas as pd

SCRIPT_DIR = Path.cwd()

SAVE_FOLDER = SCRIPT_DIR / 'data' / 'models'
os.makedirs(SAVE_FOLDER, exist_ok=True) 

features_file = os.path.join(SAVE_FOLDER, "GAonly_selected_features18.pkl")
with open(features_file, "wb") as f:
    pickle.dump(list(selected_features), f)

print(f"GA-selected features saved to '{features_file}'")

model_file = os.path.join(SAVE_FOLDER, "bestGAonly_rf_pipeline18.pkl")
joblib.dump(best_model, model_file)
print(f"Trained model saved to '{model_file}'")

GA-selected features saved to 'C:\Users\tanyi\Downloads\FYP_Processed_Data\GAonly_selected_features18.pkl'
Trained model saved to 'C:\Users\tanyi\Downloads\FYP_Processed_Data\bestGAonly_rf_pipeline18.pkl'
